# Telco Customer Churn Prediction
**Objective:** Build a machine learning model to predict customer churn.

**Business Context:** The baseline model struggles to identify actual churners due to class imbalance. By tuning class weights and decision thresholds, we optimize for **Recall**. In a real-world scenario, it is more cost-effective for a retention team to over-flag potential churners (False Positives) than to miss customers who are actually leaving (False Negatives).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score

import warnings
warnings.filterwarnings('ignore')

## 1. Data Cleaning & Feature Engineering

In [ ]:
# Load the dataset
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Handle formatting issues in TotalCharges and drop the resulting nulls
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(inplace=True)

# Binarize the target variable
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Drop customerID (no predictive value) and one-hot encode categorical features
df.drop('customerID', axis=1, inplace=True)
df = pd.get_dummies(df, drop_first=True)

print("Data cleaning complete. Prepared dataset shape:", df.shape)

## 2. Baseline XGBoost Model
First, we establish a baseline to understand how the model handles the imbalanced target variable out-of-the-box.

In [ ]:
# Define features and target
X = df.drop('Churn', axis=1)
y = df['Churn']

# 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the baseline model
model = XGBClassifier(random_state=42, eval_metric='logloss')
model.fit(X_train, y_train)

# Evaluate
predictions = model.predict(X_test)
print("Baseline Accuracy:", round(accuracy_score(y_test, predictions), 4))
print("\n--- Baseline Classification Report ---")
print(classification_report(y_test, predictions))

## 3. Optimizing for High Recall
The baseline model only achieves ~50% recall for Class 1 (Churn). To fix this, we apply a `scale_pos_weight` of 3 to penalize missed churners heavily, and we lower the prediction threshold to 0.3.

In [ ]:
# Retrain with adjusted class weights
model_tuned = XGBClassifier(scale_pos_weight=3, random_state=42, eval_metric='logloss')
model_tuned.fit(X_train, y_train)

# Apply custom threshold to maximize recall
probs = model_tuned.predict_proba(X_test)[:, 1]
custom_preds = (probs >= 0.3).astype(int)

print("--- Tuned Classification Report (Optimized for Recall) ---")
print(classification_report(y_test, custom_preds))

## 4. Feature Importance
Identifying which variables contribute most to customer churn.

In [ ]:
# Extract and plot the top 10 features driving model predictions
feat_importances = pd.Series(model_tuned.feature_importances_, index=X.columns)
plt.figure(figsize=(10, 6))
feat_importances.nlargest(10).plot(kind='barh', color='darkred')
plt.title('Top 10 Factors Driving Customer Churn (XGBoost)')
plt.xlabel('Relative Importance')
plt.gca().invert_yaxis()  # Highest importance at the top
plt.show()